[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline_solutions.ipynb)

# 01. 텍스트 분류 기준선 — 연습 문제 해설

[01_text_baseline.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline.ipynb) 끝의 연습 문제 6개에 대한 정답 코드와 해설입니다.
**먼저 직접 시도해본 뒤** 참고하세요.

> 아래 숫자는 `random_state=42` 기준입니다. 여러분의 실행 결과와 소수점 이하가 다를 수 있습니다.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
BASE_URL = "https://raw.githubusercontent.com/karzit/temp/master/notebooks/text-classification-practice/data"

if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib seaborn koreanize-matplotlib
    for _f in ["02_train.csv", "02_test_x.csv", "02_test_y.csv"]:
        !wget -q -O {_f} {BASE_URL}/{_f}
    DATA_DIR = "."
else:
    DATA_DIR = os.path.join("..", "data") if os.path.isdir(os.path.join("..", "data")) else "."

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42

train = pd.read_csv(os.path.join(DATA_DIR, "02_train.csv"))
train_clean = train.dropna(subset=["상품명"]).drop_duplicates().reset_index(drop=True)

X, y = train_clean["상품명"], train_clean["카테고리"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)


def vectorizer():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))


print("학습", len(X_train), "· 검증", len(X_valid))

---

## 문제 1. 같은 상품명에 다른 카테고리가 붙은 행

In [ ]:
n_cats = train_clean.groupby("상품명")["카테고리"].nunique()
충돌 = n_cats[n_cats > 1].index

print("서로 다른 카테고리가 붙은 상품명:", len(충돌), "종")
print("관련된 행:", len(train_clean[train_clean["상품명"].isin(충돌)]), "건")
train_clean[train_clean["상품명"].isin(충돌)].sort_values("상품명")

**이 데이터에서는 1종(2행)뿐입니다.** 상품명이 브랜드·수식어·용량까지 붙어 길기 때문에
같은 문자열이 두 번 나오는 일 자체가 드뭅니다.

**그래도 이 점검을 먼저 하는 이유**는, 이런 행이 발견되면 그것이 **라벨 오류의 직접적인 증거**이기 때문입니다.
같은 글자에 다른 정답이 붙어 있으면 둘 중 하나는 틀렸고, 모델은 그 둘을 구분할 방법이 **원리적으로 없습니다**.
실제 쇼핑몰 데이터(같은 상품을 여러 판매자가 올림)에서는 수백 건씩 나오기도 합니다.

**뺄까요?** 건수가 적으면 그대로 둬도 성능에 영향이 없습니다. 많다면 ① 둘 다 버리거나
② 더 자주 나오는 쪽으로 통일하는데, **판단하기 전에 몇 건인지부터 세는 것**이 순서입니다.
`drop_duplicates(subset=["상품명"])`을 무심코 쓰면 이 행들이 **소리 없이** 하나로 합쳐져,
문제가 있었다는 사실 자체를 못 보게 됩니다.

---

## 문제 2. `min_df` / `max_df`로 사전 줄이기

In [ ]:
for min_df in [1, 2, 3]:
    for max_df in [1.0, 0.5]:
        pipe = make_pipeline(
            TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3), min_df=min_df, max_df=max_df),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        ).fit(X_train, y_train)
        acc = accuracy_score(y_valid, pipe.predict(X_valid))
        n_features = pipe[:-1].transform(X_train[:1]).shape[1]
        print(f"min_df={min_df}, max_df={max_df}: 정확도 {acc:.4f}  피처 {n_features:,}개")

**정확도는 그대로인데 피처는 2,193 → 1,790개로 줄었습니다.**

- `min_df=3`은 "세 문서 미만에 나온 조각은 버린다"는 뜻입니다. 그렇게 드문 조각은 대개
  오타나 특이한 브랜드 표기라, 학습 데이터에서만 통하고 새 데이터에는 안 나옵니다
- `max_df=0.5`는 이 데이터에서 **아무것도 걸러내지 않았습니다**(피처 수가 그대로). 전체 문서의 절반 이상에
  나오는 글자 조각이 없기 때문입니다. 문서가 긴 데이터(뉴스·리뷰)에서는 효과가 큽니다

**피처를 줄여서 얻는 것**은 세 가지입니다. ① 학습·예측이 빨라진다, ② 모델 파일이 작아진다,
③ **드문 조각을 외우는 과적합이 줄어든다.** 성능이 같다면 **작은 쪽을 고르는 것이 맞습니다.**

---

## 문제 3. `class_weight="balanced"`

In [ ]:
for cw in [None, "balanced"]:
    pipe = make_pipeline(
        vectorizer(),
        LogisticRegression(max_iter=1000, class_weight=cw, random_state=RANDOM_STATE),
    ).fit(X_train, y_train)
    pred = pipe.predict(X_valid)
    report = classification_report(y_valid, pred, output_dict=True, zero_division=0)
    print(f"class_weight={cw}")
    print(f"  정확도 {accuracy_score(y_valid, pred):.4f} · macro f1 {f1_score(y_valid, pred, average='macro'):.4f}")
    print(f"  시리얼/영양바 재현율 {report['시리얼/영양바']['recall']:.2f}"
          f" · 즉석밥/간편식 재현율 {report['즉석밥/간편식']['recall']:.2f}")

**소수 카테고리의 재현율은 올라가고, 전체 정확도는 조금 내려갑니다.**

| | 정확도 | 시리얼/영양바 재현율 | 즉석밥/간편식 재현율 |
|---|---|---|---|
| 기본 | 0.9067 | 0.84 | 0.86 |
| `balanced` | 0.9037 | 0.88 | 0.92 |

`class_weight="balanced"`는 **건수가 적은 카테고리의 오답에 더 큰 벌점**을 매깁니다.
모델은 소수 카테고리를 더 자주 예측하게 되고, 그만큼 다수 카테고리를 조금 더 틀립니다.
**놓친 것(재현율)을 줄이는 대신 잘못 넣는 것(정밀도)이 늘어나는 거래**입니다.

**시험 기준이 정확도라면 쓰지 않는 것이 유리합니다.** 채점이 정확도 하나라면 다수 카테고리를
잘 맞히는 쪽이 점수가 높습니다. 반대로 실무에서 "모든 카테고리를 고르게 잘 분류해야 한다"면
(예: 카테고리별 담당자가 따로 있는 경우) `balanced`가 맞습니다.
**지표를 먼저 정하고 옵션을 고르는 것**이지, 그 반대가 아닙니다.

---

## 문제 4. `GridSearchCV`로 벡터화 설정과 `C` 함께 탐색

In [ ]:
pipe = make_pipeline(
    TfidfVectorizer(analyzer="char_wb"),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)

param_grid = {
    "tfidfvectorizer__ngram_range": [(2, 3), (2, 4), (1, 3)],
    "logisticregression__C": [1, 5, 20],
}

grid = GridSearchCV(pipe, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)   # 검증 데이터는 넣지 않습니다

print("최적 조합:", grid.best_params_)
print("교차 검증 점수: %.4f" % grid.best_score_)
print("검증 세트 점수: %.4f" % accuracy_score(y_valid, grid.predict(X_valid)))

**`Pipeline`의 파라미터 이름은 `단계이름__파라미터`입니다.** `make_pipeline`이 붙이는 단계 이름은
클래스명을 소문자로 바꾼 것(`tfidfvectorizer`, `logisticregression`)이고,
`pipe.get_params().keys()`로 확인할 수 있습니다.

여기서 중요한 것은 **`GridSearchCV`에 `X_train`만 넣었다는 점**입니다. 전체 데이터로 탐색하면
검증 세트가 하이퍼파라미터 선택에 관여해 **점수가 부풀려집니다**
([데이터 누출](https://github.com/karzit/temp/blob/master/glossary.md#data-leakage)).

또 하나. **교차 검증 점수(0.9137)가 검증 세트 점수(0.9067)보다 높습니다.**
`best_score_`는 "여러 조합 중 가장 잘 나온 값"이라 구조적으로 낙관적입니다.
**최종 성능은 탐색에 쓰지 않은 데이터에서 다시 재야 합니다.** 탐색 점수를 그대로 보고하는 것은
`tabular-ml-practice` 03번 연습 문제 6번에서도 나왔던 전형적인 실수입니다.

성능이 거의 오르지 않은 것도 눈여겨보세요. **이 문제에서는 하이퍼파라미터 튜닝의 여지가 작습니다.**
남은 오답 대부분이 핵심어가 없거나 라벨이 틀린 행이라, `C` 값으로는 손댈 수 없습니다.

---

## 문제 5. 확신도가 낮은 예측만 따로 보기

In [ ]:
model = make_pipeline(vectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
model.fit(X_train, y_train)

pred = model.predict(X_valid)
확신도 = model.predict_proba(X_valid).max(axis=1)

낮은순 = np.argsort(확신도)
하위100, 나머지 = 낮은순[:100], 낮은순[100:]

print("전체            정확도 %.4f" % accuracy_score(y_valid, pred))
print("확신도 하위 100건 정확도 %.4f (확신도 %.2f 이하)"
      % (accuracy_score(y_valid.values[하위100], pred[하위100]), 확신도[하위100].max()))
print("나머지          정확도 %.4f" % accuracy_score(y_valid.values[나머지], pred[나머지]))

**확신도가 낮은 100건의 정확도는 0.55, 나머지는 0.95입니다.**

모델이 내놓는 확률은 **자기가 얼마나 못 미더운지를 꽤 정직하게 알려줍니다.**
이것이 실무에서 쓰이는 방식이 있습니다.

- **사람에게 넘기기(휴먼 인 더 루프).** 확신도 0.4 미만인 건만 사람이 확인하면,
  전체의 10%만 검수하고도 오답의 상당 부분을 잡아냅니다
- **"기타"로 보류.** 자동 분류 시스템에서 확신 없는 건을 억지로 분류하지 않고 대기열에 둡니다
- **라벨 점검 대상 고르기.** 확신도가 낮은 데이터는 애매하거나 라벨이 잘못된 데이터일 확률이 높습니다

다만 **시험에서는 다릅니다.** 모든 행에 답을 적어야 하고 비워두면 그 행은 무조건 오답이므로,
확신이 없어도 가장 그럴듯한 카테고리를 채워 넣습니다.

---

## 문제 6. 브랜드를 지우면?

In [ ]:
def drop_brand(s):
    """상품명의 첫 단어(브랜드)를 떼어낸다."""
    words = s.split()
    return " ".join(words[1:]) if len(words) > 1 else s


without_brand = make_pipeline(
    vectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
).fit(X_train.map(drop_brand), y_train)

print("브랜드 포함: %.4f" % accuracy_score(y_valid, model.predict(X_valid)))
print("브랜드 제거: %.4f" % accuracy_score(y_valid, without_brand.predict(X_valid.map(drop_brand))))
print("\n예:", X_train.iloc[0], "→", drop_brand(X_train.iloc[0]))

**거의 차이가 없습니다(0.9067 → 0.9047).** 좋은 신호입니다.

이 데이터에서 브랜드는 카테고리와 무관하게 배정되어 있으므로(생성 스크립트를 보면
브랜드 14개를 모든 카테고리에 무작위로 씁니다), 브랜드는 **아무 정보가 없는 단어**입니다.
성능이 그대로라는 것은 **모델이 브랜드를 외우고 있지 않았다**는 뜻입니다.

만약 브랜드를 지우자 성능이 크게 떨어졌다면 두 가지 해석이 가능합니다.

1. **진짜 신호였다** — 실제로 라면만 만드는 회사가 있다면 브랜드는 유용한 피처입니다
2. **외우고 있었다** — 학습 데이터에만 있는 브랜드-카테고리 조합을 통째로 기억한 것.
   이 경우 새 브랜드가 들어오면 성능이 무너집니다

둘을 구분하려면 **학습 데이터에 없던 브랜드로 시험해보면** 됩니다.
5절에서 계수 상위 단어를 확인했던 것도 같은 목적의 점검이었습니다.

---

## 정리

| 문제 | 배운 것 |
|---|---|
| 1 | 같은 입력에 다른 라벨이 붙은 행은 **라벨 오류의 직접 증거**. `drop_duplicates`가 이것을 감출 수 있다 |
| 2 | `min_df`로 **성능은 유지하면서 피처를 줄일 수 있다** |
| 3 | `class_weight="balanced"`는 소수 카테고리 재현율과 전체 정확도를 맞바꾼다. **지표를 먼저 정한다** |
| 4 | `Pipeline` 전체를 탐색할 수 있다. **`best_score_`는 최종 성능이 아니다** |
| 5 | 예측 확률은 **어디를 사람이 봐야 하는지** 알려준다 |
| 6 | 특정 단어를 지워보면 **모델이 무엇에 기대고 있었는지** 알 수 있다 |